In [1]:
import os

# check the root folder of the dataset
print(os.listdir("/kaggle/input/ocular-disease-recognition-odir5k/preprocessed_images")[:5])

# check images in part 1
print(os.listdir("/kaggle/input/ocular-disease-recognition-odir5k/ODIR-5K/ODIR-5K/Training Images")[:5])

# check images in part 2
print(os.listdir("/kaggle/input/ocular-disease-recognition-odir5k/ODIR-5K/ODIR-5K/Testing Images")[:5])

['3419_left.jpg', '4176_right.jpg', '3370_left.jpg', '1255_right.jpg', '660_left.jpg']
['3419_left.jpg', '4176_right.jpg', '679_left.jpg', '3370_left.jpg', '1255_right.jpg']
['3552_right.jpg', '4711_left.jpg', '3534_right.jpg', '1911_right.jpg', '4691_left.jpg']


# PREPROCESSING  

In [2]:
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

GPU available: True
GPU name: Tesla P100-PCIE-16GB


In [3]:
import pandas as pd

df = pd.read_csv("/kaggle/input/ocular-disease-recognition-odir5k/full_df.csv")
df.head()

,ID,Patient Age,Patient Sex,Left-Fundus,Right-Fundus,Left-Diagnostic Keywords,Right-Diagnostic Keywords,N,D,G,C,A,H,M,O,filepath,labels,target,filename
0,0,69,Female,0_left.jpg,0_right.jpg,cataract,normal fundus,0,0,0,1,0,0,0,0,../input/ocular-disease-recognition-odir5k/ODI...,['N'],"[1, 0, 0, 0, 0, 0, 0, 0]",0_right.jpg
1,1,57,Male,1_left.jpg,1_right.jpg,normal fundus,normal fundus,1,0,0,0,0,0,0,0,../input/ocular-disease-recognition-odir5k/ODI...,['N'],"[1, 0, 0, 0, 0, 0, 0, 0]",1_right.jpg
2,2,42,Male,2_left.jpg,2_right.jpg,laser spot，moderate non proliferative retinopathy,moderate non proliferative retinopathy,0,1,0,0,0,0,0,1,../input/ocular-disease-recognition-odir5k/ODI...,['D'],"[0, 1, 0, 0, 0, 0, 0, 0]",2_right.jpg
3,4,53,Male,4_left.jpg,4_right.jpg,macular epiretinal membrane,mild nonproliferative retinopathy,0,1,0,0,0,0,0,1,../input/ocular-disease-recognition-odir5k/ODI...,['D'],"[0, 1, 0, 0, 0, 0, 0, 0]",4_right.jpg
4,5,50,Female,5_left.jpg,5_right.jpg,moderate non proliferative retinopathy,moderate non proliferative retinopathy,0,1,0,0,0,0,0,0,../input/ocular-disease-recognition-odir5k/ODI...,['D'],"[0, 1, 0, 0, 0, 0, 0, 0]",5_right.jpg


In [4]:


# Columns representing diseases
disease_cols = ['N','D','G','C','A','H','M','O']

# Count number of patients per disease
disease_counts = df[disease_cols].sum().sort_values()

print("Number of patients per disease (ascending order):")
print(disease_counts)


Number of patients per disease (ascending order):
H     203
M     306
A     319
G     397
C     402
O    1588
N    2101
D    2123
dtype: int64


In [5]:
df.isnull().sum()

ID                           0
Patient Age                  0
Patient Sex                  0
Left-Fundus                  0
Right-Fundus                 0
Left-Diagnostic Keywords     0
Right-Diagnostic Keywords    0
N                            0
D                            0
G                            0
C                            0
A                            0
H                            0
M                            0
O                            0
filepath                     0
labels                       0
target                       0
filename                     0
dtype: int64

In [6]:
df.drop(columns=[	'ID',	'Patient Age',	'Patient Sex',	'Left-Fundus',	'Right-Fundus',	'Left-Diagnostic Keywords',	'Right-Diagnostic Keywords', 'labels',	'target',	'filename' ])

,N,D,G,C,A,H,M,O,filepath
0,0,0,0,1,0,0,0,0,../input/ocular-disease-recognition-odir5k/ODI...
1,1,0,0,0,0,0,0,0,../input/ocular-disease-recognition-odir5k/ODI...
2,0,1,0,0,0,0,0,1,../input/ocular-disease-recognition-odir5k/ODI...
3,0,1,0,0,0,0,0,1,../input/ocular-disease-recognition-odir5k/ODI...
4,0,1,0,0,0,0,0,0,../input/ocular-disease-recognition-odir5k/ODI...
...,...,...,...,...,...,...,...,...,...
6387,0,1,0,0,0,0,0,0,../input/ocular-disease-recognition-odir5k/ODI...
6388,0,1,0,0,0,0,0,0,../input/ocular-disease-recognition-odir5k/ODI...
6389,0,1,0,0,0,0,0,0,../input/ocular-disease-recognition-odir5k/ODI...
6390,0,1,0,0,0,0,0,0,../input/ocular-disease-recognition-odir5k/ODI...


In [7]:
IMAGE_DIR = "/kaggle/input/ocular-disease-recognition-odir5k/preprocessed_images"

existing_images = os.listdir(IMAGE_DIR)
existing_images[:10], len(existing_images)

(['3419_left.jpg',
  '4176_right.jpg',
  '3370_left.jpg',
  '1255_right.jpg',
  '660_left.jpg',
  '484_right.jpg',
  '4221_right.jpg',
  '2396_left.jpg',
  '543_left.jpg',
  '3017_left.jpg'],
 6392)

In [8]:
IMAGE_DIR = "/kaggle/input/ocular-disease-recognition-odir5k/preprocessed_images"

# IMPORTANT: extract only the filename
df['filepath'] = df['filename'].apply(
    lambda x: os.path.join(IMAGE_DIR, os.path.basename(x))
)

In [9]:
missing = [p for p in df['filepath'].values if not os.path.exists(p)]
len(missing)

0

In [10]:
from sklearn.model_selection import train_test_split

X = df['filepath'].values
y = df[['N','D','G','C','A','H','M','O']].values

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.30,   # 30% validation
    random_state=42,
    shuffle=True
)

In [11]:
len(X_train)

4474

In [12]:
len(X_val)

1918

# AUGMENTATION

In [13]:
import torchvision.transforms as transforms

# 🔹 Training transforms (Resize + Normalize + Augmentation)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),                 # Step 1: Resize images
    transforms.RandomHorizontalFlip(p=0.5),        # Step 2: Horizontal flip
    transforms.RandomRotation(15),                 # Step 3: Small rotations
    transforms.ColorJitter(brightness=0.1,
                           contrast=0.1),          # Step 4: Light intensity changes
    transforms.ToTensor(),                         # Step 5: Convert to tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406],# Step 6: Normalize (ImageNet)
                         std=[0.229, 0.224, 0.225])
])

# 🔹 Validation / Test transforms (NO augmentation)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),                 # Step 1: Resize images
    transforms.ToTensor(),                         # Step 2: Convert to tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406],# Step 3: Normalize
                         std=[0.229, 0.224, 0.225])
])

# DATASET 

In [14]:
import torch
from torch.utils.data import Dataset
from PIL import Image

In [15]:
class FundusDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None): #constructor that contains image loc, labels and augmnetaion pipeline 
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self): # retuen dataset size as PyTorch needs this to know how many samples exist, decide when one epoch ends, work with DataLoader

        return len(self.image_paths)

    def __getitem__(self, idx): # Defines how to fetch ONE data sample.
        # Load image
        image = Image.open(self.image_paths[idx]).convert("RGB")

        # Get labels
        label = torch.tensor(self.labels[idx], dtype=torch.float32) # float is required for BCEWithLogitsLoss / focal / asymmetric losses

        # Apply transform
        if self.transform:
            image = self.transform(image)

        return image, label

# DATALOADER

In [16]:
from torch.utils.data import DataLoader

# Create Dataset objects
train_dataset = FundusDataset(X_train, y_train, transform=train_transform)
test_dataset  = FundusDataset(X_val, y_val, transform=test_transform)

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=16, #with batch size 16 model will learn the avg trend and will not oversmooth results 
    shuffle=True,
    num_workers=2 #num_workers controls how many CPU processes load and preprocess data in parallel, preventing the GPU from becoming idle during training
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2
)

print("Train batches:", len(train_loader))
print("Test batches:", len(test_loader))

Train batches: 280
Test batches: 120


# MODEL ARCHITECTURE

**CBAM**

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F


In [18]:
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(),
            nn.Linear(in_channels // reduction, in_channels, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.size()
        avg_out = self.fc(self.avg_pool(x).view(b, c))
        max_out = self.fc(self.max_pool(x).view(b, c))
        out = avg_out + max_out
        return x * self.sigmoid(out).view(b, c, 1, 1)

In [19]:
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        return x * self.sigmoid(self.conv(x_cat))

In [20]:
class CBAM(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.ca = ChannelAttention(channels)
        self.sa = SpatialAttention()

    def forward(self, x):
        x = self.ca(x)
        x = self.sa(x)
        return x

In [21]:
from torchvision import models

In [22]:
class EfficientNet_CBAM(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()

        # Load pretrained EfficientNet-B0
        self.backbone = models.efficientnet_b0(pretrained=True)

        # Extract feature dimension
        in_features = self.backbone.classifier[1].in_features

        # Remove default classifier
        self.backbone.classifier = nn.Identity()

        # CBAM block
        self.cbam = CBAM(in_features)

        # Classification head
        self.classifier = nn.Linear(in_features, num_classes)

    def forward(self, x):
        # Feature extraction
        x = self.backbone.features(x)     # (B, C, H, W)

        # Attention
        x = self.cbam(x)

        # Global Average Pooling
        x = F.adaptive_avg_pool2d(x, 1)
        x = x.view(x.size(0), -1)

        # Multi-label output (logits)
        x = self.classifier(x)

        return x

In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [24]:
model = EfficientNet_CBAM(num_classes=8)
model = model.to(device)

print(model)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 164MB/s]


EfficientNet_CBAM(
  (backbone): EfficientNet(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
      (1): Sequential(
        (0): MBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
              (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
              (2): SiLU(inplace=True)
            )
            (1): SqueezeExcitation(
              (avgpool): AdaptiveAvgPool2d(output_size=1)
              (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
              (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
              (activation): SiLU(inplace=True)
              (

In [25]:
#managing class imbalance
import torch
import numpy as np

label_cols = ['N','D','G','C','A','H','M','O']

# Convert labels to numpy
labels_np = df[label_cols].values

# Count positives per class
pos_counts = labels_np.sum(axis=0)

# Total samples
num_samples = labels_np.shape[0]

# Count negatives per class
neg_counts = num_samples - pos_counts

# pos_weight = negatives / positives
pos_weight = neg_counts / pos_counts

pos_weight = torch.tensor(pos_weight, dtype=torch.float32)

print("Positive counts :", pos_counts)
print("Negative counts :", neg_counts)
print("pos_weight      :", pos_weight)

Positive counts : [2101 2123  397  402  319  203  306 1588]
Negative counts : [4291 4269 5995 5990 6073 6189 6086 4804]
pos_weight      : tensor([ 2.0424,  2.0108, 15.1008, 14.9005, 19.0376, 30.4877, 19.8889,  3.0252])


In [26]:
#freezing layer in efficient net
for param in model.backbone.parameters():
    param.requires_grad = False

In [27]:
#updated loss function
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

#optimizer (define Adam only for trainable parameters)
import torch.optim as optim

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=3e-5,
    weight_decay=1e-4
)

In [28]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()   # enable training mode
    total_loss = 0.0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        # Forward
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    return avg_loss

In [29]:
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    return avg_loss

In [30]:
num_epochs = 15
unfreeze=3
best_val_loss = float("inf")

for epoch in range(1, num_epochs + 1):
    print(f"\n🚀 Epoch {epoch}/{num_epochs}")

    if epoch==unfreeze:
        print("unfreezing")

    for param in model.backbone.parameters():
        param.requires_grad=True

    optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5,
    weight_decay=1e-4 )
    
    
    train_loss = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss = validate(
        model,
        test_loader,
        criterion,
        device
    )

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val   Loss: {val_loss:.4f}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pth")
        print("✅ Best model saved")


🚀 Epoch 1/15
Train Loss: 1.1521
Val   Loss: 1.1618
✅ Best model saved

🚀 Epoch 2/15
Train Loss: 1.0895
Val   Loss: 1.0770
✅ Best model saved

🚀 Epoch 3/15
unfreezing
Train Loss: 1.0226
Val   Loss: 1.0130
✅ Best model saved

🚀 Epoch 4/15
Train Loss: 0.9765
Val   Loss: 0.9667
✅ Best model saved

🚀 Epoch 5/15
Train Loss: 0.9326
Val   Loss: 0.9340
✅ Best model saved

🚀 Epoch 6/15
Train Loss: 0.9031
Val   Loss: 0.9049
✅ Best model saved

🚀 Epoch 7/15
Train Loss: 0.8761
Val   Loss: 0.8820
✅ Best model saved

🚀 Epoch 8/15
Train Loss: 0.8530
Val   Loss: 0.8653
✅ Best model saved

🚀 Epoch 9/15
Train Loss: 0.8288
Val   Loss: 0.8543
✅ Best model saved

🚀 Epoch 10/15
Train Loss: 0.8065
Val   Loss: 0.8361
✅ Best model saved

🚀 Epoch 11/15
Train Loss: 0.7955
Val   Loss: 0.8259
✅ Best model saved

🚀 Epoch 12/15
Train Loss: 0.7797
Val   Loss: 0.8160
✅ Best model saved

🚀 Epoch 13/15
Train Loss: 0.7613
Val   Loss: 0.8085
✅ Best model saved

🚀 Epoch 14/15
Train Loss: 0.7504
Val   Loss: 0.8006
✅ Best mo

In [31]:
torch.save({
    'epoch': epoch,
    'model_state': model.state_dict(),
    'optimizer_state': optimizer.state_dict(),
    'best_val_loss': best_val_loss
}, 'checkpoint.pth')


In [32]:
checkpoint = torch.load("checkpoint.pth", map_location=device)

model.load_state_dict(checkpoint["model_state"])
optimizer.load_state_dict(checkpoint["optimizer_state"])

start_epoch = checkpoint["epoch"] + 1
best_val_loss = checkpoint["best_val_loss"]
